
# Maximal Non-Branching Paths

**Topic:** Graph Algorithms and Genome Assembly

### Overview

This notebook implements the Maximal Non-Branching Paths algorithm for directed graphs, a fundamental approach used in graph-based genome assembly.

A node is classified as **1-in-1-out** when it has exactly one incoming edge and one outgoing edge.

A maximal non-branching path contains internal nodes that are 1-in-1-out, while its endpoints correspond to branching nodes or path boundaries. Isolated cycles consisting entirely of 1-in-1-out nodes are also considered.

### Objectives

- Parse directed graphs from adjacency lists.
- Calculate node in-degrees and out-degrees.
- Identify 1-in-1-out nodes.
- Extract maximal non-branching paths.
- Detect isolated cycles and self-loops.
- Preserve repeated edges.
- Validate graph traversal and input parsing.

### Implementation

The algorithm first identifies paths starting at nodes that are not 1-in-1-out. Each path is extended through intermediate 1-in-1-out nodes until it reaches a branching point or endpoint.

A second traversal identifies isolated cycles that were not included in the initial paths.

Individual edges are tracked using their source node and adjacency-list position, allowing parallel edges to be preserved.

### Learning Context

This implementation was developed as part of the *Genome Sequencing* course in the Bioinformatics Specialization offered by UC San Diego on Coursera.

The notebook was subsequently reviewed to improve edge tracking, isolated-cycle detection, input validation, and automated testing.

In [ ]:
from collections import defaultdict, Counter

def parse_graph_from_input(input_text):
    """
    Parse a directed graph from an adjacency-list string.

    Each non-empty line must contain an integer node,
    followed by a colon and space-separated integer neighbors.

    Example
    -------
    1: 2 3
    2: 4

    Returns
    -------
    defaultdict(list)
        Directed adjacency list preserving repeated edges.

    Raises
    ------
    ValueError
        If a non-empty line has an invalid format.
    """
    graph = defaultdict(list)

    if not isinstance(input_text, str):
        raise ValueError("Graph input must be a string.")

    for line in input_text.splitlines():
        line = line.strip()

        if not line:
            continue

        if ":" not in line:
            raise ValueError(f"Invalid adjacency-list line: {line}")

        node_text, neighbors_text = line.split(":", 1)

        try:
            node = int(node_text.strip())
            neighbors = [int(value) for value in neighbors_text.split()]
        except ValueError as exc:
            raise ValueError(
                f"Invalid node or neighbor in line: {line}") from exc

        graph[node].extend(neighbors)

    return graph


def in_degree(graph):
    """Calculate the in-degree of each node in the directed graph."""
    indeg = defaultdict(int)
    for node in graph:
        for neighbor in graph[node]:
            indeg[neighbor] += 1
    return indeg


def out_degree(graph):
    """Calculate the out-degree of each node in the directed graph."""
    outdeg = defaultdict(int)
    for node in graph:
        outdeg[node] = len(graph[node])
    return outdeg


def is_1_in_1_out(node, indeg, outdeg):
    """Return True if a node has exactly one incoming and one outgoing edge."""
    return indeg[node] == 1 and outdeg[node] == 1


def maximal_non_branching_paths(graph):
    """
    Find all maximal non-branching paths and isolated cycles
    in a directed graph.

    Each edge is identified by its source node and position
    in the adjacency list, preserving parallel edges.

    Parameters
    ----------
    graph : dict
        Adjacency list representing a directed graph.

    Returns
    -------
    list[list[int]]
        Maximal non-branching paths represented as node sequences.
    """
    paths = []
    indeg = in_degree(graph)
    outdeg = out_degree(graph)
    visited_edges = set()

    # Step 1: Find maximal non-branching paths
    for node in graph:
        if not is_1_in_1_out(node, indeg, outdeg) and outdeg[node] > 0:
            for edge_index, neighbor in enumerate(graph[node]):
                path = [node, neighbor]
                visited_edges.add((node, edge_index))

                while is_1_in_1_out(neighbor, indeg, outdeg):
                    next_node = graph[neighbor][0]
                    visited_edges.add((neighbor, 0))
                    path.append(next_node)
                    neighbor = next_node

                paths.append(path)

    # Step 2: Find isolated cycles
    for node in graph:
        if is_1_in_1_out(node, indeg, outdeg) and (node, 0) not in visited_edges:
            cycle = [node]
            current_node = node

            while (current_node, 0) not in visited_edges:
                visited_edges.add((current_node, 0))

                next_node = graph[current_node][0]
                cycle.append(next_node)
                current_node = next_node

            paths.append(cycle)

    return paths


# Example adjacency-list input
input_text = """1: 2
2: 3
3: 4 5
7: 6
6: 7"""

# Parse the graph
graph = parse_graph_from_input(input_text)

# Find maximal non-branching paths
paths = maximal_non_branching_paths(graph)

# Display the resulting paths
for path in paths:
    print(" -> ".join(map(str, path)))

1 -> 2 -> 3
3 -> 4
3 -> 5
7 -> 6 -> 7


## Validation and Testing

The implementation was evaluated using seven tests covering path identification, edge conservation, cycle detection, and input parsing.

The tests include:

1. **Basic maximal non-branching paths:** Identification of non-branching paths and an isolated cycle.
2. **Repeated-edge handling:** Verification that parallel edges are preserved.
3. **Repeated-edge regression:** Confirmation that duplicate edges remain represented as separate paths.
4. **Original example regression:** Verification that the reference example produces the expected paths.
5. **Isolated self-loop:** Detection of a cycle containing a single node.
6. **Edge conservation:** Comparison of original and reconstructed edge frequencies using `Counter`.
7. **Input parsing and validation:** Verification of flexible adjacency-list formatting and rejection of malformed inputs.

All seven tests passed.

### Notes

* The algorithm preserves edge multiplicity and returns paths as sequences of integer node identifiers.
* Isolated cycles are represented as closed paths, with the starting node repeated at the end.
* The implementation is intended for educational use and illustrates concepts relevant to graph-based genome assembly.

In [16]:
# Test 1: Basic maximal non-branching paths
input_text = """1: 2
2: 3
3: 4 5
7: 6
6: 7"""

graph = parse_graph_from_input(input_text)
paths = maximal_non_branching_paths(graph)

print("Test 1 passed: Basic maximal non-branching paths")
for path in paths:
    print(" -> ".join(map(str, path)))


# Test 2: Repeated edges diagnostic
input_text = """1: 2 2
2: 3"""

graph = parse_graph_from_input(input_text)
paths = maximal_non_branching_paths(graph)

print("Test 2 passed: Repeated edges preserved")
print("Generated paths:", paths)
print("Expected edge count:", 3)
print("Generated edge count:", sum(len(path) - 1 for path in paths))


# Test 3: Verify repeated-edge preservation
graph = parse_graph_from_input("1: 2 2\n2: 3")
paths = maximal_non_branching_paths(graph)

assert sorted(paths) == [[1, 2], [1, 2], [2, 3]]

print("Test 3 passed: Repeated edges preserved")
print("Generated paths:", paths)


# Test 4: Verify the original example
input_text = """1: 2
2: 3
3: 4 5
7: 6
6: 7"""

graph = parse_graph_from_input(input_text)
paths = maximal_non_branching_paths(graph)

assert sorted(paths) == sorted([[1, 2, 3], [3, 4], [3, 5], [7, 6, 7]])

print("Test 4 passed: Original example preserved")


# Test 5: Isolated self-loop
graph = {1: [1]}
paths = maximal_non_branching_paths(graph)

assert paths == [[1, 1]]

print("Test 5 passed: Isolated self-loop detected")
print("Generated paths:", paths)


# Test 6: Verify edge conservation
graph = {1: [2, 2], 2: [3], 3: [4, 5], 6: [7], 7: [6]}

paths = maximal_non_branching_paths(graph)
original_edges = Counter((node, neighbor) for node,
                         neighbors in graph.items() for neighbor in neighbors)
reconstructed_edges = Counter((path[i], path[i + 1])
                              for path in paths for i in range(len(path) - 1))

assert reconstructed_edges == original_edges

print("Test 6 passed: All graph edges preserved")
print("Original edge count:", sum(original_edges.values()))
print("Reconstructed edge count:", sum(reconstructed_edges.values()))


# Test 7: Validate adjacency-list parsing
input_text = """1:2 2
2: 3

3:
"""

graph = parse_graph_from_input(input_text)

assert dict(graph) == {1: [2, 2], 2: [3], 3: []}

invalid_inputs = ["1 2 3", "A: 2", "1: B", 123]

for invalid_input in invalid_inputs:
    try:
        parse_graph_from_input(invalid_input)
    except ValueError:
        continue
    raise AssertionError(f"Invalid input was accepted: {invalid_input}")

print("Test 7 passed: Adjacency-list parsing and validation")


Test 1 passed: Basic maximal non-branching paths
1 -> 2 -> 3
3 -> 4
3 -> 5
7 -> 6 -> 7
Test 2 passed: Repeated edges preserved
Generated paths: [[1, 2], [1, 2], [2, 3]]
Expected edge count: 3
Generated edge count: 3
Test 3 passed: Repeated edges preserved
Generated paths: [[1, 2], [1, 2], [2, 3]]
Test 4 passed: Original example preserved
Test 5 passed: Isolated self-loop detected
Generated paths: [[1, 1]]
Test 6 passed: All graph edges preserved
Original edge count: 7
Reconstructed edge count: 7
Test 7 passed: Adjacency-list parsing and validation
